In [20]:
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt 

import numpy as np 
import tensorflow as tf 
from tensorflow.keras import * 
from tensorflow.keras.layers import * 
from tensorflow.keras.models import load_model, save_model 
import tensorflow.keras.utils as utils
from keras.datasets import mnist 


### Original Floating-point Model 

In [21]:
(x_train, y_train_num), (x_test, y_test_num) = mnist.load_data() 

def Net(): 
    inputs = Input(shape=x_train.shape[1:])
    x = Conv2D(4, kernel_size=(3,3), strides=(1,1), padding='same')(inputs)
    x = ReLU()(x)
    x = MaxPool2D((2,2),strides=(2,2), padding="same")(x)

    x = Flatten()(x)
    x = Dense(20)(x)

    x = ReLU()(x)
    x = Dense(10)(x)
    predictions = Softmax()(x)
    return Model(inputs=inputs, outputs=predictions)

In [18]:
"""
quantize numpy array
return:
q_x: quantized array
fp_x: reverse quantized array to float
"""
def quantize_array(x, bit_depth=8):
    min_x = x.min() 
    max_x = x.max()

    #find number of integer bits to represent this range
    int_bits = int(np.ceil(np.log2(max(abs(min_x),abs(max_x)))))
    
    if int_bits < 0: int_bits = 0

    frac_bits = bit_depth - 1 - int_bits #remaining bits are fractional bits (1-bit for sign)

    #floating point weights are scaled and rounded to [-2^(bit-1),2^(bit-1)-1], which are used in 
    #the fixed-point operations on the actual hardware (i.e., microcontroller)
    q_x = np.round(x*(2**frac_bits))

    #To quantify the impact of quantized weights, scale them back to
    # original range to run inference using quantized weights
    fp_x = q_x/(2**frac_bits)
    
    return q_x, fp_x, int_bits, frac_bits


"""
compute frac_bits based on max and min list
used for activation(feature map) quantization
"""
def min_max_quantize(max_list, min_list, bit_depth=8):
    frac_list = []
    for i in range(len(max_list)):
        int_bits = int(np.ceil(np.log2(max(abs(min_list[i]),abs(max_list[i])))))
        if int_bits < 0: int_bits = 0
        frac_bits = bit_depth - 1 - int_bits #remaining bits are fractional bits (1-bit for sign)
        frac_list.append(frac_bits)
    return frac_list